# 02 – Phase 2: Baseline Benchmarking & Profiling

Runs the complete benchmark pipeline in one shot:
1. Segmentation accuracy (Dice, IoU, pixel accuracy, per-class)
2. Model complexity (FLOPs, params, size)
3. Inference performance (latency, FPS)
4. Memory usage (RAM, GPU)
5. CPU utilisation
6. Energy consumption
7. CSV + Markdown report generation
8. Visualisation plots

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/wet-amd-segmentation-edge/project')

import torch
from utils import Config, set_seed, get_device, device_info
from utils.dataset import build_dataloaders
from models.model_loader import load_model, SegmentationInference
from models.baseline_model import CLASS_NAMES

from evaluation.metrics import compute_all_metrics, MetricAccumulator
from evaluation.profiler import ModelProfiler
from evaluation.energy_monitor import EnergyMonitor, print_energy_results
from evaluation.visualization import (
    plot_predictions, plot_per_class_dice, plot_metric_comparison
)
from evaluation.report_generator import (
    generate_markdown_report, save_full_results_csv, print_summary_table
)

cfg    = Config()
cfg.ensure_dirs()
set_seed(cfg.seed)

device = get_device()
print('Device:', device_info(device))

In [ ]:
# ── Load model & data ────────────────────────────────────────────────────────
model = load_model(cfg, device=device)
inf   = SegmentationInference(model, device)

_, _, test_loader = build_dataloaders(cfg)
print(f'Test batches: {len(test_loader)}')

In [ ]:
# ── 1. Segmentation Accuracy ─────────────────────────────────────────────────
print('Running segmentation evaluation on test set ...')

acc = MetricAccumulator()
sample_images, sample_gt, sample_pred = [], [], []

model.eval()
with torch.no_grad():
    for images, masks in test_loader:
        probs, pred_mask = inf.predict(images)
        acc.update(compute_all_metrics(pred_mask, masks))
        if len(sample_images) < 4:
            sample_images.append(images)
            sample_gt.append(masks)
            sample_pred.append(pred_mask)

accuracy_metrics = acc.mean()

print(f"\nMean Dice : {accuracy_metrics['mean_dice']:.4f}")
print(f"Mean IoU  : {accuracy_metrics['mean_iou']:.4f}")
print(f"Pixel Acc : {accuracy_metrics['pixel_acc']:.4f}")
print('\nPer-class Dice:')
for name in CLASS_NAMES:
    key   = f"dice_{name.lower().replace(' ', '_')}"
    score = accuracy_metrics.get(key, 0.0)
    bar   = '█' * int(score * 30)
    print(f'  {name:<18} {score:.4f}  {bar}')

In [ ]:
# ── 2. Model Complexity & Performance Profiling ───────────────────────────────
print('Running profiling ...')

profiler = ModelProfiler(cfg, device)
profiling_metrics = profiler.run(
    model,
    label='baseline_fp32',
    n_runs=cfg.benchmark_runs,
    warmup=cfg.benchmark_warmup,
)
ModelProfiler.print_report(profiling_metrics)

In [ ]:
# ── 3. Energy Monitoring ─────────────────────────────────────────────────────
print('Running energy monitoring ...')

monitor = EnergyMonitor()
print(f'Backend: {monitor.backend_name}')

dummy_input = torch.randn(1, cfg.in_channels, *cfg.image_size).to(device)

with monitor.track('baseline_inference') as energy_metrics:
    model.eval()
    with torch.no_grad():
        for _ in range(50):   # 50 inferences
            _ = model(dummy_input)

# Normalise to per-inference
if energy_metrics.get('energy_j', 0) > 0:
    energy_metrics['energy_mj_per_inference'] = (
        energy_metrics['energy_j'] * 1000 / 50
    )

print_energy_results(energy_metrics)

In [ ]:
# ── 4. Generate Reports ───────────────────────────────────────────────────────
print_summary_table(accuracy_metrics, profiling_metrics, energy_metrics)

md_path  = generate_markdown_report(
    accuracy_metrics, profiling_metrics, energy_metrics,
    cfg, label='baseline_fp32'
)
csv_path = save_full_results_csv(
    accuracy_metrics, profiling_metrics, energy_metrics, cfg
)

print(f'Markdown report → {md_path}')
print(f'CSV results     → {csv_path}')

In [ ]:
# ── 5. Visualisation ──────────────────────────────────────────────────────────
# Qualitative predictions
plot_predictions(
    torch.cat(sample_images),
    torch.cat(sample_gt),
    torch.cat(sample_pred),
    cfg, n=4, show=True
)

# Per-class Dice bar chart
class_dice_scores = [
    accuracy_metrics.get(f"dice_{n.lower().replace(' ', '_')}", 0.0)
    for n in CLASS_NAMES
]
plot_per_class_dice(class_dice_scores, cfg, show=True)

# Summary metric comparison (placeholder for future PTQ/QAT comparison)
plot_metric_comparison(
    labels=['baseline_fp32'],
    metrics={
        'Mean Dice': [accuracy_metrics['mean_dice']],
        'Mean IoU':  [accuracy_metrics['mean_iou']],
        'Pixel Acc': [accuracy_metrics['pixel_acc']],
    },
    cfg=cfg,
    show=True,
)